<a href="https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
import os
if not os.path.exists('/content/flyrank-ml-internship'):
    !git clone https://github.com/him2079/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.install_extension("httpfs")
con.load_extension("httpfs")
con.execute(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');""")

/content/flyrank-ml-internship


## 1. Method choice and why

Method: Random Forest classifier, compared against Logistic Regression as a simpler baseline model. My lane (refresh/CTR opportunity scoring) is fundamentally about ranking pages by a combination of interacting signals (impressions, clicks, position, CTR gap) — a single linear boundary may miss the kind of threshold effects and interactions I saw in the baseline data (e.g. the extreme zero-click-despite-good-position pages). Random Forest handles nonlinear interactions and gives permutation importance for free, which supports the error-analysis requirement in Section 4. I'm not reaching for Gradient Boosting yet, since the brief warns against rewarding complexity alone — Random Forest is the simplest step up from the linear baseline that can plausibly capture the pattern.

In [7]:
model_df = con.execute("""
WITH early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
),
late AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) as clicks_late
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1,2
)
SELECT e.*, l.clicks_late,
    CASE WHEN l.clicks_late < e.clicks THEN 1 ELSE 0 END as is_declining
FROM early e
JOIN late l ON e.client_hash_id = l.client_hash_id AND e.content_hash_id = l.content_hash_id
""").df()

import numpy as np
model_df['ctr'] = model_df['clicks'] / model_df['impressions']
model_df['position_tier'] = np.select(
    [model_df['avg_position'] <= 3, model_df['avg_position'] <= 10, model_df['avg_position'] <= 20],
    ['pos_1-3', 'pos_4-10', 'pos_11-20'],
    default='pos_20+'
)
tier_median_ctr = model_df.groupby('position_tier')['ctr'].transform('median')
model_df['ctr_gap_score'] = np.clip(1 - (model_df['ctr'] / tier_median_ctr), 0, 1)
model_df['baseline_score'] = model_df['ctr_gap_score'] * np.log1p(model_df['impressions'])

model_df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(92548, 11)

## 2. Split design

Client-grouped split. Pages from the same client likely share writing style, template, and traffic patterns — if the same client's pages appeared in both train and test, the model could partially memorize client-specific quirks rather than learning generalizable signal, inflating the score. A grouped split keeps every page from a given client entirely on one side, matching the "client-holdout" approach the starter pipeline itself used. Confirmed: 32 clients in train, 8 in test, zero overlap between the two groups.

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

print("Train clients:", train_df['client_hash_id'].nunique(), "Test clients:", test_df['client_hash_id'].nunique())
print("Overlap check (should be 0):", len(set(train_df['client_hash_id']) & set(test_df['client_hash_id'])))

Train clients: 32 Test clients: 8
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline

On the same client-holdout test split, at Precision@50: the Week 4 baseline rule scored 0.18, logistic regression scored 0.74, and random forest scored 0.62. Both models clearly beat the baseline — a ~3-4x improvement — but logistic regression outperformed random forest here, which is not what I expected going in. A likely explanation: with only 32 training clients and a relatively small, low-dimensional feature set (5 features), random forest's extra flexibility may be overfitting to noise that logistic regression's simpler linear boundary avoids. This is a useful lesson — more complexity doesn't automatically mean a better model, especially with a modest amount of training data. I'm treating logistic regression as the stronger model for this lane going forward, not random forest, despite it being the "fancier" choice.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

features = ['impressions', 'clicks', 'avg_position', 'ctr', 'ctr_gap_score']

X_train, y_train = train_df[features].fillna(0), train_df['is_declining']
X_test, y_test = test_df[features].fillna(0), test_df['is_declining']

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

results = {
    'baseline (Week 4 rule)': precision_at_k(y_test, test_df['baseline_score'].values, k=50),
    'logistic regression': precision_at_k(y_test, lr_scores, k=50),
    'random forest': precision_at_k(y_test, rf_scores, k=50),
}
import pandas as pd
pd.DataFrame(results.items(), columns=['method', 'precision_at_50'])

,method,precision_at_50
0,baseline (Week 4 rule),0.18
1,logistic regression,0.74
2,random forest,0.64


## 4. Errors and interpretation

What the model leans on: CTR is by far the dominant coefficient (29.8, more than 7x the next largest), meaning low CTR is the strongest predictor of decline — which makes intuitive sense, since CTR captures actual audience behavior directly, while position and impressions are more indirect signals. ctr_gap_score (the tier-adjusted version) has a smaller negative coefficient — worth noting this seems almost redundant with raw CTR once both are in the model, and a cleaner version of this model might drop one of the two to avoid near-duplicate signal.

Error pattern: Looking at the top 20 highest-risk predictions, 15 of 20 correctly matched is_declining == 1 — consistent with the 0.74 Precision@50 score. The 5 misses (rows where is_declining == 0 but the model predicted high risk) share a pattern: they tend to have somewhat higher click counts relative to their impressions than the correct predictions around them (e.g. 465 clicks on 74,133 impressions, or 194 clicks on 48,661 impressions) — meaning the model may be slightly over-weighting raw CTR without fully separating "genuinely low CTR" from "CTR that's low but still respectable given real click volume." A next iteration could add a feature capturing absolute click volume alongside the rate, not just the rate itself.

In [10]:
coef_importance = pd.Series(lr.coef_[0], index=features).sort_values(key=abs, ascending=False)
print(coef_importance)

test_df['lr_score'] = lr_scores
top_predictions = test_df.sort_values('lr_score', ascending=False).head(20)
top_predictions[['impressions','clicks','avg_position','ctr','is_declining','lr_score']]

ctr              29.826061
ctr_gap_score    -3.872157
avg_position     -0.046603
clicks           -0.010007
impressions       0.000151
dtype: float64


,impressions,clicks,avg_position,ctr,is_declining,lr_score
9575,91474.0,185.0,4.103429,0.002022,1,0.999997
19329,108663.0,493.0,3.040838,0.004537,1,0.999995
19231,83715.0,133.0,3.453361,0.001589,1,0.999994
19203,86860.0,51.0,5.785512,0.000587,1,0.999980
13472,73639.0,18.0,2.786744,0.000244,0,0.999705
9225,60089.0,200.0,3.042033,0.003328,1,0.999594
10098,74133.0,465.0,3.152695,0.006273,0,0.999363
9405,51103.0,186.0,3.019148,0.003640,1,0.998645
18571,44610.0,61.0,4.284483,0.001367,1,0.998490
9451,47063.0,164.0,2.335783,0.003485,1,0.998055


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.